# 中证800 vs 中证500 LGB 模型确认实验

目的：在上一版因子稳定性对比之后，进一步确认 universe 选择在模型层面的影响。

固定条件：

- 数据：使用上一版因子稳定性实验产出的固定面板 `csi800_vs_csi500_factor_stability_outputs/csi800_vs_csi500_monthly_factor_panel.csv`。
- 因子：固定 37 个 JQ factor。
- 标签：`alpha_1m`。
- 模型：LightGBM regression，固定迭代数，无 early stopping。
- 评估窗口：rolling fixed train-end 年度 OOS。

核心对比：

1. `train_csi800_test_csi800`：中证800训练，中证800测试。
2. `train_csi500_test_csi500`：中证500训练，中证500测试。
3. `train_csi800_test_csi500`：中证800训练，只在中证500测试，用来判断宽池训练是否泛化到500。
4. `train_csi500_test_csi800`：中证500训练，中证800测试，用来判断500模型是否外推到宽池。

这个 notebook 只确认模型层 universe 差异，不做导出 pkl，不修改回测文件。

In [ ]:
# =========================
# Config
# =========================
import os
import gc
import math
import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
    LGB_IMPORT_OK = True
except Exception as _lgb_err:
    LGB_IMPORT_OK = False
    LGB_IMPORT_ERROR = _lgb_err

OUT_DIR = "csi800_vs_csi500_lgb_model_confirm_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SOURCE_PANEL_PATH = "csi800_vs_csi500_factor_stability_outputs/csi800_vs_csi500_monthly_factor_panel.csv"
TARGET_COL = "alpha_1m"
RAW_RETURN_COL = "raw_return_1m"
DATE_COL = "rebalance_date"
STOCK_COL = "stock"

TOP_LIST = [10, 20, 30]

YEARLY_WINDOWS = [
    {"window_name": "train2016_2022_test2023", "train_start": "2016-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"window_name": "train2017_2023_test2024", "train_start": "2017-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"window_name": "train2018_2024_test2025", "train_start": "2018-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"window_name": "train2019_2025_test2026", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]

MODEL_CASES = [
    {"strategy_name": "train_csi800_test_csi800", "train_universe": "csi800", "test_universe": "csi800"},
    {"strategy_name": "train_csi500_test_csi500", "train_universe": "csi500", "test_universe": "csi500"},
    {"strategy_name": "train_csi800_test_csi500", "train_universe": "csi800", "test_universe": "csi500"},
    {"strategy_name": "train_csi500_test_csi800", "train_universe": "csi500", "test_universe": "csi800"},
]

FEATURE_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield",
    "sales_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
    "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit", "operating_profit_per_share",
    "net_operate_cash_flow_per_share", "total_operating_revenue_per_share",
    "ACCA", "growth", "net_working_capital", "super_quick_ratio", "MLEV",
    "debt_to_equity_ratio", "debt_to_tangible_equity_ratio",
    "momentum", "Rank1M", "sharpe_ratio_60", "Variance20", "liquidity", "beta",
    "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC", "Skewness20", "Kurtosis20", "Kurtosis60",
]

LGB_PARAMS = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
    "seed": 20260616,
    "feature_fraction_seed": 20260616,
    "bagging_seed": 20260616,
    "data_random_seed": 20260616,
}
NUM_BOOST_ROUND = 120

print("LGB import ok:", LGB_IMPORT_OK)
if not LGB_IMPORT_OK:
    print("LightGBM import error:", LGB_IMPORT_ERROR)
print("source panel:", SOURCE_PANEL_PATH)
print("feature count:", len(FEATURE_COLS))

In [ ]:
# =========================
# Load panel
# =========================
def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col])
    return out


def require_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError("missing columns: " + ",".join(missing))


if not os.path.exists(SOURCE_PANEL_PATH):
    raise IOError("fixed source panel not found: " + SOURCE_PANEL_PATH + " . Run 中证800_vs_中证500_因子稳定性对比实验.ipynb first.")

panel_df = pd.read_csv(SOURCE_PANEL_PATH)
panel_df = safe_to_datetime(panel_df, ["rebalance_date", "feature_date", "next_date"])
require_columns(panel_df, ["universe_name", STOCK_COL, DATE_COL, TARGET_COL, RAW_RETURN_COL] + FEATURE_COLS)

for col in FEATURE_COLS + [TARGET_COL, RAW_RETURN_COL]:
    panel_df[col] = pd.to_numeric(panel_df[col], errors="coerce")

panel_df = panel_df.replace([np.inf, -np.inf], np.nan)
print("loaded panel:", panel_df.shape)
print(panel_df.groupby("universe_name")[DATE_COL].agg(["min", "max", "nunique"]))
print(panel_df.groupby("universe_name")[STOCK_COL].count())

In [ ]:
# =========================
# Model and evaluation helpers
# =========================
def fit_fill_values(train_df, feature_cols):
    values = {}
    for col in feature_cols:
        s = train_df[col].replace([np.inf, -np.inf], np.nan)
        med = s.median()
        values[col] = 0.0 if pd.isnull(med) else float(med)
    return values


def make_feature_matrix(df, feature_cols, fill_values):
    x = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan)
    x = x.fillna(pd.Series(fill_values)).fillna(0.0)
    return x[feature_cols]


def train_lgb_model(train_df, feature_cols, target_col):
    if not LGB_IMPORT_OK:
        raise RuntimeError("LightGBM is unavailable in this environment")
    train_df = train_df.copy()
    train_df = train_df[train_df[target_col].notnull()].copy()
    fill_values = fit_fill_values(train_df, feature_cols)
    x_train = make_feature_matrix(train_df, feature_cols, fill_values)
    y_train = train_df[target_col].astype(float).values
    dtrain = lgb.Dataset(x_train, label=y_train, free_raw_data=True)
    model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND)
    return model, fill_values


def predict_scores(model, fill_values, df, feature_cols):
    x = make_feature_matrix(df, feature_cols, fill_values)
    return np.asarray(model.predict(x)).reshape(-1)


def rank_ic_by_month(df, score_col, target_col):
    rows = []
    for dt, gdf in df.groupby(DATE_COL):
        tmp = gdf[[score_col, target_col]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(tmp) < 40 or tmp[score_col].nunique() <= 2:
            ic = np.nan
        else:
            ic = tmp[score_col].rank().corr(tmp[target_col].rank())
        rows.append({"rebalance_date": dt, "rank_ic": ic, "sample_count": len(gdf), "valid_count": len(tmp)})
    return pd.DataFrame(rows)


def calc_max_drawdown(cum_ret):
    s = pd.Series(cum_ret).astype(float)
    if s.empty:
        return np.nan
    wealth = 1.0 + s
    roll_max = wealth.cummax()
    dd = wealth / roll_max - 1.0
    return dd.min()


def evaluate_topn(score_df, score_col, topn):
    rows = []
    prev_targets = None
    for dt, gdf in score_df.groupby(DATE_COL):
        gdf = gdf.replace([np.inf, -np.inf], np.nan).dropna(subset=[score_col, RAW_RETURN_COL, TARGET_COL]).copy()
        if gdf.empty:
            continue
        top = gdf.sort_values(score_col, ascending=False).head(topn).copy()
        targets = list(top[STOCK_COL])
        bench_ret = gdf[RAW_RETURN_COL].mean()
        raw_ret = top[RAW_RETURN_COL].mean()
        alpha_ret = raw_ret - bench_ret
        target_set = set(targets)
        if prev_targets is None or topn == 0:
            turnover = np.nan
        else:
            turnover = 1.0 - float(len(target_set.intersection(prev_targets))) / float(topn)
        rows.append({
            "rebalance_date": dt,
            "topn": topn,
            "target_count": len(top),
            "raw_return_1m": raw_ret,
            "benchmark_mean_return_1m": bench_ret,
            "excess_return_1m": alpha_ret,
            "alpha_label_mean_1m": top[TARGET_COL].mean(),
            "turnover": turnover,
            "targets": ",".join(targets),
        })
        prev_targets = target_set
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values("rebalance_date")
        out["cum_ret"] = (1.0 + out["raw_return_1m"]).cumprod() - 1.0
        out["cum_benchmark"] = (1.0 + out["benchmark_mean_return_1m"]).cumprod() - 1.0
        out["cum_excess"] = (1.0 + out["excess_return_1m"]).cumprod() - 1.0
    return out


def summarize_monthly(monthly_df):
    if monthly_df.empty:
        return {}
    return {
        "months": len(monthly_df),
        "cum_ret": monthly_df["cum_ret"].iloc[-1],
        "cum_benchmark": monthly_df["cum_benchmark"].iloc[-1],
        "cum_excess": monthly_df["cum_excess"].iloc[-1],
        "mean_monthly_excess": monthly_df["excess_return_1m"].mean(),
        "win_rate": (monthly_df["excess_return_1m"] > 0).mean(),
        "max_drawdown": calc_max_drawdown(monthly_df["cum_ret"]),
        "avg_turnover": monthly_df["turnover"].dropna().mean(),
        "drop_top1_excess": monthly_df.sort_values("excess_return_1m", ascending=False).iloc[1:]["excess_return_1m"].add(1.0).prod() - 1.0 if len(monthly_df) > 1 else np.nan,
        "drop_top3_excess": monthly_df.sort_values("excess_return_1m", ascending=False).iloc[3:]["excess_return_1m"].add(1.0).prod() - 1.0 if len(monthly_df) > 3 else np.nan,
    }

In [ ]:
# =========================
# Run one yearly window and all model cases
# =========================
def slice_by_universe_and_dates(df, universe_name, start_date, end_date):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    out = df[(df["universe_name"] == universe_name) & (df[DATE_COL] >= start) & (df[DATE_COL] <= end)].copy()
    return out


def run_one_case(panel, window_spec, case_spec):
    train_df = slice_by_universe_and_dates(panel, case_spec["train_universe"], window_spec["train_start"], window_spec["train_end"])
    test_df = slice_by_universe_and_dates(panel, case_spec["test_universe"], window_spec["test_start"], window_spec["test_end"])
    if train_df.empty or test_df.empty:
        print("skip empty", window_spec["window_name"], case_spec["strategy_name"], train_df.shape, test_df.shape)
        return None, None, None, None

    model, fill_values = train_lgb_model(train_df, FEATURE_COLS, TARGET_COL)
    score_df = test_df.copy()
    score_df["score"] = predict_scores(model, fill_values, score_df, FEATURE_COLS)
    score_df["window_name"] = window_spec["window_name"]
    score_df["strategy_name"] = case_spec["strategy_name"]
    score_df["train_universe"] = case_spec["train_universe"]
    score_df["test_universe"] = case_spec["test_universe"]
    score_df["train_start"] = window_spec["train_start"]
    score_df["train_end"] = window_spec["train_end"]

    ic_df = rank_ic_by_month(score_df, "score", TARGET_COL)
    ic_df["window_name"] = window_spec["window_name"]
    ic_df["strategy_name"] = case_spec["strategy_name"]
    ic_df["train_universe"] = case_spec["train_universe"]
    ic_df["test_universe"] = case_spec["test_universe"]

    monthly_parts = []
    summary_rows = []
    for topn in TOP_LIST:
        monthly = evaluate_topn(score_df, "score", topn)
        if monthly.empty:
            continue
        monthly["window_name"] = window_spec["window_name"]
        monthly["strategy_name"] = case_spec["strategy_name"]
        monthly["train_universe"] = case_spec["train_universe"]
        monthly["test_universe"] = case_spec["test_universe"]
        monthly_parts.append(monthly)

        sm = summarize_monthly(monthly)
        sm.update({
            "window_name": window_spec["window_name"],
            "strategy_name": case_spec["strategy_name"],
            "train_universe": case_spec["train_universe"],
            "test_universe": case_spec["test_universe"],
            "topn": topn,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "rank_ic_mean": ic_df["rank_ic"].mean(),
            "rank_ic_ir": ic_df["rank_ic"].mean() / ic_df["rank_ic"].std() if ic_df["rank_ic"].std() > 0 else np.nan,
        })
        summary_rows.append(sm)

    monthly_df = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
    summary_df = pd.DataFrame(summary_rows)
    meta_df = pd.DataFrame([{
        "window_name": window_spec["window_name"],
        "strategy_name": case_spec["strategy_name"],
        "train_universe": case_spec["train_universe"],
        "test_universe": case_spec["test_universe"],
        "train_start": window_spec["train_start"],
        "train_end": window_spec["train_end"],
        "test_start": window_spec["test_start"],
        "test_end": window_spec["test_end"],
        "train_rows": len(train_df),
        "test_rows": len(test_df),
        "feature_count": len(FEATURE_COLS),
        "num_boost_round": NUM_BOOST_ROUND,
        "params": str(LGB_PARAMS),
    }])

    del model, train_df, test_df
    gc.collect()
    return score_df, monthly_df, summary_df, ic_df, meta_df

In [ ]:
# =========================
# Run experiment
# =========================
score_parts = []
monthly_parts = []
summary_parts = []
ic_parts = []
meta_parts = []

for window_spec in YEARLY_WINDOWS:
    for case_spec in MODEL_CASES:
        print("running", window_spec["window_name"], case_spec["strategy_name"])
        score, monthly, summary, ic, meta = run_one_case(panel_df, window_spec, case_spec)
        if score is not None and not score.empty:
            keep_cols = ["window_name", "strategy_name", "train_universe", "test_universe", DATE_COL, "feature_date", "next_date", STOCK_COL, "market_segment", RAW_RETURN_COL, TARGET_COL, "score"]
            score_parts.append(score[[c for c in keep_cols if c in score.columns]].copy())
        if monthly is not None and not monthly.empty:
            monthly_parts.append(monthly)
        if summary is not None and not summary.empty:
            summary_parts.append(summary)
        if ic is not None and not ic.empty:
            ic_parts.append(ic)
        if meta is not None and not meta.empty:
            meta_parts.append(meta)

score_df = pd.concat(score_parts, ignore_index=True, sort=False) if score_parts else pd.DataFrame()
monthly_df = pd.concat(monthly_parts, ignore_index=True, sort=False) if monthly_parts else pd.DataFrame()
summary_df = pd.concat(summary_parts, ignore_index=True, sort=False) if summary_parts else pd.DataFrame()
ic_df = pd.concat(ic_parts, ignore_index=True, sort=False) if ic_parts else pd.DataFrame()
meta_df = pd.concat(meta_parts, ignore_index=True, sort=False) if meta_parts else pd.DataFrame()

summary_df = summary_df.sort_values(["topn", "cum_excess", "rank_ic_mean"], ascending=[True, False, False]) if not summary_df.empty else summary_df

score_df.to_csv(os.path.join(OUT_DIR, "lgb_score.csv"), index=False)
monthly_df.to_csv(os.path.join(OUT_DIR, "lgb_monthly.csv"), index=False)
summary_df.to_csv(os.path.join(OUT_DIR, "lgb_summary.csv"), index=False)
ic_df.to_csv(os.path.join(OUT_DIR, "lgb_rank_ic.csv"), index=False)
meta_df.to_csv(os.path.join(OUT_DIR, "lgb_model_meta.csv"), index=False)

print("score:", score_df.shape)
print("monthly:", monthly_df.shape)
print("summary:", summary_df.shape)
summary_df.head(30)

In [ ]:
# =========================
# Cross-window comparison tables
# =========================
def build_cross_window_summary(summary):
    if summary.empty:
        return pd.DataFrame()
    rows = []
    for (strategy_name, topn), gdf in summary.groupby(["strategy_name", "topn"]):
        rows.append({
            "strategy_name": strategy_name,
            "topn": topn,
            "windows": len(gdf),
            "avg_cum_excess": gdf["cum_excess"].mean(),
            "median_cum_excess": gdf["cum_excess"].median(),
            "min_cum_excess": gdf["cum_excess"].min(),
            "positive_windows": (gdf["cum_excess"] > 0).sum(),
            "avg_rank_ic_mean": gdf["rank_ic_mean"].mean(),
            "avg_rank_ic_ir": gdf["rank_ic_ir"].mean(),
            "avg_win_rate": gdf["win_rate"].mean(),
            "avg_max_drawdown": gdf["max_drawdown"].mean(),
            "avg_turnover": gdf["avg_turnover"].mean(),
            "avg_drop_top1_excess": gdf["drop_top1_excess"].mean(),
            "avg_drop_top3_excess": gdf["drop_top3_excess"].mean(),
        })
    out = pd.DataFrame(rows)
    return out.sort_values(["topn", "avg_cum_excess", "avg_rank_ic_mean"], ascending=[True, False, False])


def build_window_pair_delta(summary, topn=10):
    if summary.empty:
        return pd.DataFrame()
    pivot = summary[summary["topn"] == topn].pivot_table(index="window_name", columns="strategy_name", values="cum_excess", aggfunc="first")
    out = pivot.reset_index()
    if "train_csi800_test_csi800" in out.columns and "train_csi500_test_csi500" in out.columns:
        out["csi800_minus_csi500_same_universe"] = out["train_csi800_test_csi800"] - out["train_csi500_test_csi500"]
    if "train_csi800_test_csi500" in out.columns and "train_csi500_test_csi500" in out.columns:
        out["wide_train_minus_500_train_on_500"] = out["train_csi800_test_csi500"] - out["train_csi500_test_csi500"]
    if "train_csi500_test_csi800" in out.columns and "train_csi800_test_csi800" in out.columns:
        out["500_train_minus_wide_train_on_800"] = out["train_csi500_test_csi800"] - out["train_csi800_test_csi800"]
    return out

cross_summary_df = build_cross_window_summary(summary_df)
delta_top10_df = build_window_pair_delta(summary_df, topn=10)
delta_top20_df = build_window_pair_delta(summary_df, topn=20)

cross_summary_df.to_csv(os.path.join(OUT_DIR, "lgb_cross_window_summary.csv"), index=False)
delta_top10_df.to_csv(os.path.join(OUT_DIR, "lgb_delta_top10.csv"), index=False)
delta_top20_df.to_csv(os.path.join(OUT_DIR, "lgb_delta_top20.csv"), index=False)

print("=== cross window summary ===")
print(cross_summary_df.to_string(index=False))
print("\n=== top10 delta ===")
print(delta_top10_df.to_string(index=False))
print("\n=== top20 delta ===")
print(delta_top20_df.to_string(index=False))

In [ ]:
# =========================
# Segment contribution for CSI800 test strategies
# =========================
def build_segment_contribution(score_data, topn=10):
    rows = []
    if score_data.empty or "market_segment" not in score_data.columns:
        return pd.DataFrame()
    for (window_name, strategy_name, dt), gdf in score_data.groupby(["window_name", "strategy_name", DATE_COL]):
        if "test_csi800" not in strategy_name:
            continue
        tmp = gdf.dropna(subset=["score", RAW_RETURN_COL, TARGET_COL]).copy()
        if tmp.empty:
            continue
        top = tmp.sort_values("score", ascending=False).head(topn)
        for seg, sdf in top.groupby("market_segment"):
            rows.append({
                "window_name": window_name,
                "strategy_name": strategy_name,
                "rebalance_date": dt,
                "topn": topn,
                "market_segment": seg,
                "count": len(sdf),
                "raw_return_1m": sdf[RAW_RETURN_COL].mean(),
                "alpha_1m": sdf[TARGET_COL].mean(),
            })
    return pd.DataFrame(rows)

segment_top10_df = build_segment_contribution(score_df, topn=10)
segment_top20_df = build_segment_contribution(score_df, topn=20)
segment_top10_df.to_csv(os.path.join(OUT_DIR, "lgb_segment_contribution_top10.csv"), index=False)
segment_top20_df.to_csv(os.path.join(OUT_DIR, "lgb_segment_contribution_top20.csv"), index=False)

print("segment top10 rows:", segment_top10_df.shape)
if not segment_top10_df.empty:
    seg_summary = segment_top10_df.groupby(["strategy_name", "market_segment"]).agg(
        avg_count=("count", "mean"),
        avg_alpha=("alpha_1m", "mean"),
        avg_raw_return=("raw_return_1m", "mean"),
    ).reset_index()
    seg_summary.to_csv(os.path.join(OUT_DIR, "lgb_segment_summary_top10.csv"), index=False)
    print(seg_summary.to_string(index=False))

In [ ]:
# =========================
# Final reading guide
# =========================
print("Outputs saved to:", OUT_DIR)
for name in [
    "lgb_summary.csv",
    "lgb_cross_window_summary.csv",
    "lgb_delta_top10.csv",
    "lgb_delta_top20.csv",
    "lgb_rank_ic.csv",
    "lgb_monthly.csv",
    "lgb_segment_summary_top10.csv",
]:
    print(" -", os.path.join(OUT_DIR, name))

print("\nDecision guide:")
print("1. If train_csi500_test_csi500 wins most windows but train_csi800_test_csi500 is close, CSI800 training is enough and 500 can be a sleeve.")
print("2. If train_csi500_test_csi500 clearly beats train_csi800_test_csi800 and has lower drawdown/turnover, consider a CSI500 mainline candidate.")
print("3. If train_csi500_test_csi800 performs poorly, CSI500 model is style-specific and should not replace the CSI800 baseline.")
print("4. If train_csi800_test_csi800 is close or better across Top10/Top20, keep CSI800 as the robust-system base.")